# 01 — 数据协议审计（开发用）

独立重建 BANKING77 和 CLINC150 数据协议；数据清理、Source ID、文本归一化和采样规则保持不变。所有统计、重复文本、标签映射及协议指纹均在单元中展示，不导出 CSV/JSON。

该文件用于开发审计，不计入最终 12 份 method–dataset Notebook。最终 ZIP 只放已执行、保留输出的实验 Notebook。原始公开数据和模型可使用缓存或在线加载；CLINC JSON 缺失时仅在内存中下载并校验 SHA256。

In [1]:
# Set True only when explicitly choosing to install missing packages in this interpreter.
INSTALL_MISSING_DEPENDENCIES = False
from importlib import util, metadata
import subprocess
import sys

_requirements = {"numpy": "numpy", "pandas": "pandas", "datasets": "datasets", "sklearn": "scikit-learn"}
_missing = [package for module, package in _requirements.items() if util.find_spec(module) is None]
if _missing:
    print("Missing dependencies:", _missing)
    if not INSTALL_MISSING_DEPENDENCIES:
        raise RuntimeError("Install in the current kernel: " + sys.executable + " -m pip install " + " ".join(_missing) + "; or set INSTALL_MISSING_DEPENDENCIES=True and rerun setup")
    subprocess.check_call([sys.executable, "-m", "pip", "install", *_missing])

import hashlib
import json
import re
from collections import Counter, defaultdict
from pathlib import Path

import numpy as np
import pandas as pd
from datasets import load_dataset
from IPython.display import display
from sklearn.model_selection import train_test_split

# Resolve output paths without inspecting another notebook or results files.
cwd = Path.cwd()
PROJECT = cwd.parent if cwd.name == "notebooks" else cwd

SEED = 42
K = 8
VALIDATION = 0.10

CLINC_SHA256 = "36923c3705a59e08fe9c3883d8bc2dd966ef93e22cb78ac41171782a698d56e0"
NORMALIZATION = "strip/collapse whitespace and Unicode casefold; punctuation retained"


def normalize_text(text):
    if not isinstance(text, str) or not text.strip():
        raise ValueError("Text must be a non-empty string")
    return re.sub(r'\s+', ' ', text).strip().casefold()


def text_hash(text):
    return hashlib.sha256(normalize_text(text).encode("utf-8")).hexdigest()


def audit_overlap(left, right, left_name, right_name):
    a = left.assign(_key=left["text"].map(normalize_text))
    b = right.assign(_key=right["text"].map(normalize_text))
    # Keep the original audit column order for the saved 01 evidence.
    pairs = a.drop(columns="text_sha256", errors="ignore").merge(
        b.drop(columns="text_sha256", errors="ignore"),
        on="_key", suffixes=("_left", "_right")
    )
    report = {
        "left": left_name, "right": right_name,
        "matching_pairs": len(pairs), "unique_texts": int(pairs["_key"].nunique()),
        "left_rows": int(pairs["source_id_left"].nunique()),
        "right_rows": int(pairs["source_id_right"].nunique()),
        "conflicting_pairs": int((pairs["label_left"] != pairs["label_right"]).sum()),
    }
    return report, pairs


def remove_train_overlap(train, protected, name):
    report, pairs = audit_overlap(train, protected, name, "protected")
    drop_ids = set(pairs["source_id_left"])
    cleaned = train.loc[~train["source_id"].isin(drop_ids)].copy()
    assert len(train) - len(cleaned) == report["left_rows"]
    print(name, "removed training rows:", len(drop_ids),
          "conflicting pairs:", report["conflicting_pairs"])
    return cleaned, report, pairs


def verify_disjoint(a, b, name_a, name_b):
    overlap = set(a["text_sha256"]) & set(b["text_sha256"])
    assert not overlap, f"Normalized-text overlap: {name_a} / {name_b}"
    assert set(a["source_id"]).isdisjoint(b["source_id"])
    return {"left": name_a, "right": name_b, "unique_texts": len(overlap)}


def make_source_ids(df, dataset_name, split_name):
    prefix = "banking77" if dataset_name == "banking77" else "clinc"
    result = df.copy()
    result["source_id"] = [f"{prefix}_{split_name}_{i}" for i in range(len(result))]
    result["text_sha256"] = result["text"].map(text_hash)
    return result


def read_original_clinc():
    """Read optional local original data, or download in memory; never export JSON."""
    path = PROJECT / "data" / "clinc150" / "data_full.json"
    if path.exists():
        content = path.read_bytes()
    else:
        import urllib.request
        url = "https://raw.githubusercontent.com/clinc/oos-eval/master/data/data_full.json"
        content = urllib.request.urlopen(url, timeout=90).read()
    assert hashlib.sha256(content).hexdigest() == CLINC_SHA256, "Original CLINC SHA256 mismatch"
    return json.loads(content)


def sample_fewshot(pool, shot_budget, seed, sampling_mode):
    """Sample a few-shot set; nested budgets share per-class random prefixes."""
    if sampling_mode not in {"independent", "nested"}:
        raise ValueError("sampling_mode must be 'independent' or 'nested'")
    if shot_budget is None:
        return pool.copy()
    if type(shot_budget) is not int or shot_budget <= 0:
        raise ValueError("shot_budget must be None or a positive integer")
    if sampling_mode == "nested" and shot_budget > 16:
        raise ValueError("nested sampling supports shot budgets up to 16")
    assert pool.groupby("label").size().min() >= shot_budget, "Insufficient examples per class"
    if sampling_mode == "independent":
        # Preserve legacy pandas sample behavior, IDs and row order exactly.
        return pool.groupby("label", group_keys=False).sample(
            n=shot_budget, random_state=seed).copy()

    rng = np.random.RandomState(seed)
    chosen_ids = []
    for label in sorted(pool["label"].unique().tolist()):
        class_ids = pool.loc[pool["label"] == label, "source_id"].to_numpy()
        chosen_ids.extend(rng.permutation(class_ids)[:shot_budget].tolist())
    selected_ids = set(chosen_ids)
    return pool.loc[pool["source_id"].isin(selected_ids)].copy()


def build_protocol(dataset_name, shot_budget=8, seed=42, validation_fraction=0.10,
                   sampling_mode="independent"):
    if dataset_name not in {"banking77", "clinc150"}:
        raise ValueError("Choose banking77 or clinc150")
    if shot_budget is not None and (type(shot_budget) is not int or shot_budget <= 0):
        raise ValueError("shot_budget must be None or a positive integer")
    if sampling_mode not in {"independent", "nested"}:
        raise ValueError("sampling_mode must be 'independent' or 'nested'")
    if sampling_mode == "nested" and shot_budget is not None and shot_budget > 16:
        raise ValueError("nested sampling supports shot budgets up to 16")
    if type(seed) is not int or not 0 <= seed < 2**32:
        raise ValueError("seed must be an integer in [0, 2**32)")
    if not 0 < validation_fraction < 1:
        raise ValueError("validation_fraction must be between zero and one")
    audits = {}
    pairs = {}
    oos_audit = []
    original_sha256 = None
    if dataset_name == "banking77":
        raw = load_dataset("PolyAI/banking77")
        id_to_name = dict(enumerate(raw["train"].features["label"].names))
        expected_classes = 77
        train = make_source_ids(raw["train"].to_pandas(), dataset_name, "train")
        test = make_source_ids(raw["test"].to_pandas(), dataset_name, "test")
        cleaned, audits["train_vs_test"], pairs["train_vs_test"] = remove_train_overlap(train, test, "banking_train")
        pool, validation = train_test_split(
            cleaned, test_size=validation_fraction, random_state=seed,
            stratify=cleaned["label"]
        )
        pool, audits["pool_vs_validation"], pairs["pool_vs_validation"] = remove_train_overlap(pool, validation, "banking_pool")
        original_splits = {"train": train, "test": test}
        expected_labels = set(id_to_name)
    else:
        original = read_original_clinc()
        original_sha256 = CLINC_SHA256
        converted = load_dataset("DeepPavlov/clinc150", "default")
        id_to_names = defaultdict(set)
        original_splits = {}
        all_original_texts = {text for rows in original.values() for text, label in rows}
        for split, raw_name in [("train", "train"), ("validation", "val"), ("test", "test")]:
            raw_rows = original[raw_name]
            text_to_names = defaultdict(set)
            for text, name in raw_rows:
                text_to_names[text].add(name)
            assert all(len(names) == 1 for names in text_to_names.values()), "Ambiguous original labels"
            hf_frame = converted[split].to_pandas()
            known = hf_frame.loc[hf_frame["label"].notna()]
            assert Counter(known["utterance"]) == Counter(text for text, name in raw_rows)
            for text, number in zip(known["utterance"], known["label"]):
                id_to_names[int(number)].update(text_to_names[text])
            missing = Counter(hf_frame.loc[hf_frame["label"].isna(), "utterance"])
            oos = Counter(text for text, name in original["oos_" + raw_name])
            unknown = missing - oos
            assert not (oos - missing), "Original OOS rows missing from converted split"
            oos_audit.append({
                "split": split, "null_labels": sum(missing.values()),
                "original_oos_matches": sum((missing & oos).values()),
                "unconfirmed_extra_rows": sum(unknown.values()),
                "extra_rows_in_any_original_split": sum(n for text, n in unknown.items() if text in all_original_texts),
            })
            frame = pd.DataFrame(raw_rows, columns=["text", "label"])
            # Preserve the established IDs: i indexes original JSON in-domain rows.
            original_splits[split] = make_source_ids(frame, dataset_name, split)
        assert len(id_to_names) == 150 and all(len(names) == 1 for names in id_to_names.values())
        id_to_name = {number: next(iter(names)) for number, names in id_to_names.items()}
        assert set(id_to_name) == set(range(150)) and len(set(id_to_name.values())) == 150
        expected_classes = 150
        expected_labels = set(id_to_name.values())
        train = original_splits["train"]
        validation = original_splits["validation"]
        test = original_splits["test"]
        pool, audits["train_vs_validation"], pairs["train_vs_validation"] = remove_train_overlap(train, validation, "clinc_train")
        pool, audits["train_vs_test_after_validation"], pairs["train_vs_test_after_validation"] = remove_train_overlap(pool, test, "clinc_train")

    checks = []
    for a, b, an, bn in [
        (pool, validation, "pool", "validation"),
        (pool, test, "pool", "test"),
        (validation, test, "validation", "test"),
    ]:
        checks.append(verify_disjoint(a, b, an, bn))
    for frame, frame_name in [(pool, "pool"), (validation, "validation"), (test, "test")]:
        assert set(frame["label"]) == expected_labels, f"Unexpected labels in {frame_name}"
        assert frame["source_id"].is_unique, f"Duplicate source_id in {frame_name}"
        assert frame["text_sha256"].equals(frame["text"].map(text_hash)), f"Hash mismatch in {frame_name}"
    source = pd.concat(original_splits.values()).set_index("source_id")
    assert source.index.is_unique, "Duplicate source_id in original source rows"
    for frame in [pool, validation, test]:
        restored = source.loc[frame["source_id"], ["text", "label", "text_sha256"]].reset_index(drop=True)
        pd.testing.assert_frame_equal(restored, frame[["text", "label", "text_sha256"]].reset_index(drop=True))
    # Official protected content and row order must remain complete.
    pd.testing.assert_frame_equal(test, original_splits["test"])
    if dataset_name == "clinc150":
        pd.testing.assert_frame_equal(validation, original_splits["validation"])
    fewshot = sample_fewshot(pool, shot_budget, seed, sampling_mode)
    if shot_budget is not None:
        counts = fewshot.groupby("label").size()
        assert len(counts) == expected_classes and counts.eq(shot_budget).all(), (
            "Few-shot budget/class coverage mismatch")
    assert fewshot["source_id"].is_unique
    pool_by_id = pool.set_index("source_id")
    restored_fewshot = pool_by_id.loc[
        fewshot["source_id"], ["text", "label", "text_sha256"]
    ].reset_index(drop=True)
    pd.testing.assert_frame_equal(
        restored_fewshot,
        fewshot[["text", "label", "text_sha256"]].reset_index(drop=True),
    )
    meta = {
        "protocol_version": "2.0",
        "dataset": dataset_name, "shot_budget": shot_budget, "seed": seed,
        "sampling_mode": sampling_mode,
        "nested_max_shot": 16 if sampling_mode == "nested" else None,
        "validation_fraction": validation_fraction if dataset_name == "banking77" else None,
        "normalization": NORMALIZATION, "hash_algorithm": "SHA256 of normalized UTF-8 text",
        "split_sizes": {"pool": len(pool), "validation": len(validation), "test": len(test),
                        "fewshot": len(fewshot) if shot_budget is not None else None},
        "fewshot_note": "When shot_budget is None, fewshot is a pool copy for compatibility.",
        "classes": expected_classes, "overlap_audits": audits,
        "disjoint_checks": checks, "oos_audit": oos_audit,
        "clinc_original_sha256": original_sha256,
        "id_to_name": id_to_name,
        "source_id_order": "HF original split rows" if dataset_name == "banking77" else "Original CLINC JSON in-domain split rows (legacy protocol)",
    }
    return {"pool": pool, "validation": validation, "test": test, "fewshot": fewshot,
            "protocol_meta": meta, "audit_pairs": pairs}


def validate_training_protocol(protocol):
    """Verify selected training IDs and content using the current seed and budget."""
    meta = protocol["protocol_meta"]
    offer = protocol["pool"]
    selected = protocol["fewshot"]
    budget = meta["shot_budget"]
    mode = meta["sampling_mode"]
    seed = meta["seed"]
    if mode not in {"independent", "nested"}:
        raise ValueError("sampling_mode must be 'independent' or 'nested'")
    print(
        "Protocol metadata:",
        {"protocol_version": meta.get("protocol_version"),
         "sampling_mode": mode,
         "nested_max_shot": meta.get("nested_max_shot"),
         "split_sizes": meta.get("split_sizes")},
    )
    for name in ("pool", "validation", "test", "fewshot"):
        frame = protocol[name]
        assert frame["source_id"].is_unique, f"Duplicate source_id in {name}"
        assert frame["text_sha256"].equals(frame["text"].map(text_hash)), f"Hash mismatch in {name}"

    for left_name, right_name in [
        ("pool", "validation"), ("pool", "test"), ("validation", "test")
    ]:
        verify_disjoint(protocol[left_name], protocol[right_name], left_name, right_name)

    offer_by_id = offer.set_index("source_id")
    expected_hashes = selected["source_id"].map(offer_by_id["text_sha256"])
    assert expected_hashes.notna().all(), "Few-shot ID outside pool"
    assert selected["text_sha256"].reset_index(drop=True).equals(
        expected_hashes.reset_index(drop=True)), "Few-shot hash differs from pool"
    restored_selected = offer_by_id.loc[
        selected["source_id"], ["text", "label"]
    ].reset_index(drop=True)
    pd.testing.assert_frame_equal(
        restored_selected, selected[["text", "label"]].reset_index(drop=True)
    )

    if budget is None:
        pd.testing.assert_frame_equal(selected, offer)
    else:
        expected = sample_fewshot(offer, budget, seed, mode)
        assert set(selected["source_id"]) == set(expected["source_id"]), (
            "Few-shot IDs differ from current seed/budget/mode")
    return True




## 重建两数据集并展示统计

保留官方保护划分，训练侧移除确认重复文本。展示类别数、各划分规模、每类数量范围及重复处理证据。固定 seed=42、independent、8-shot，供后续方法保持一致。

In [2]:
banking_protocol = build_protocol("banking77", shot_budget=K, seed=SEED, validation_fraction=VALIDATION)
clinc_protocol = build_protocol("clinc150", shot_budget=K, seed=SEED, validation_fraction=VALIDATION)
rows = []
for protocol in [banking_protocol, clinc_protocol]:
    meta = protocol["protocol_meta"]
    print(json.dumps(meta, ensure_ascii=False, default=str))
    for part in ["pool", "validation", "test", "fewshot"]:
        frame = protocol[part]
        counts = frame["label"].value_counts()
        rows.append({"dataset": meta["dataset"], "split": part,
                     "sampling_mode": meta["sampling_mode"],
                     "shot_budget": meta["shot_budget"],
                     "fewshot_is_pool_copy": part == "fewshot" and meta["shot_budget"] is None,
                     "rows": len(frame), "classes": len(counts),
                     "min_per_class": counts.min(), "max_per_class": counts.max(),
                     "median_words": frame["text"].str.split().str.len().median()})
    for name, pairs in protocol["audit_pairs"].items():
        if not pairs.empty:
            print(meta["dataset"], name)
            display(pairs[["text_left", "label_left", "text_right", "label_right"]])
display(pd.DataFrame(rows).set_index(["dataset", "split"]))


banking_train removed training rows: 7 conflicting pairs: 0
banking_pool removed training rows: 0 conflicting pairs: 0
clinc_train removed training rows: 3 conflicting pairs: 2
clinc_train removed training rows: 2 conflicting pairs: 2
{"protocol_version": "2.0", "dataset": "banking77", "shot_budget": 8, "seed": 42, "sampling_mode": "independent", "nested_max_shot": null, "validation_fraction": 0.1, "normalization": "strip/collapse whitespace and Unicode casefold; punctuation retained", "hash_algorithm": "SHA256 of normalized UTF-8 text", "split_sizes": {"pool": 8996, "validation": 1000, "test": 3080, "fewshot": 616}, "fewshot_note": "When shot_budget is None, fewshot is a pool copy for compatibility.", "classes": 77, "overlap_audits": {"train_vs_test": {"left": "banking_train", "right": "protected", "matching_pairs": 7, "unique_texts": 7, "left_rows": 7, "right_rows": 7, "conflicting_pairs": 0}, "pool_vs_validation": {"left": "banking_pool", "right": "protected", "matching_pairs": 0, "

,text_left,label_left,text_right,label_right
0,\nHow do I unblock my PIN?,49,How do I unblock my PIN?,49
1,What businesses accept this card?,10,\n\nWhat businesses accept this card?,10
2,\nWhere can I use my card?,10,Where can I use my card?,10
3,There are a few transaction that I don't recog...,22,There are a few transaction that I don't recog...,22
4,\nAt which ATMs can I use this card?,3,At which ATMs can I use this card?,3
5,\nWhich cash machines will allow me to change ...,21,Which cash machines will allow me to change my...,21
6,I don't live in the UK. Can I still get a card?,24,I don't live in the UK. Can I still get a card?,24


,text_left,label_left,text_right,label_right
0,what is on my to do list,todo_list,what is on my to do list,reminder
1,turn up your volume,whisper_mode,turn up your volume,change_volume
2,hey what's up,greeting,hey what's up,greeting


,text_left,label_left,text_right,label_right
0,what's your designation,what_is_your_name,what's your designation,user_name
1,where did you grow up,how_old_are_you,where did you grow up,where_are_you_from


sampling_mode  shot_budget  ...  max_per_class  median_words
dataset   split                                  ...                             
banking77 pool         independent            8  ...            168          10.0
          validation   independent            8  ...             19          10.0
          test         independent            8  ...             40           9.0
          fewshot      independent            8  ...              8          10.0
clinc150  pool         independent            8  ...            100           8.0
          validation   independent            8  ...             20           8.0
          test         independent            8  ...             30           8.0
          fewshot      independent            8  ...              8           8.0

[8 rows x 8 columns]

## 在单元中核验并展示协议

以原有冻结记录计算的 SHA256 指纹核对 Source ID、标签、文本哈希及行序；不读取或生成审计 CSV。完整协议和标签映射保存在内存变量中，单元展示摘要与前五行，避免大量冗余输出。

In [ ]:
# Expected fingerprints of the original seed-42 independent 8-shot manifests.
EXPECTED_MANIFEST_SHA256 = {'banking77/pool': '6f58936f33ee15f8e6c7700d8b628f7025c590b082392b6aeb3695f44203fc9c', 'banking77/validation': '713010f4e8bdb5715ef8403301f9d4cefd0e016c5fd71b7b1d541c1b735effe7', 'banking77/test': '5849b28e6d50c2cb73e4bc473067434259a46834f23bbe62b1db9670f09a802a', 'banking77/fewshot': 'd6c292066bc100654e398e52fc802e02242c296accae802c383531117b6af531', 'clinc150/pool': 'd8c692b3b44cc4172f6edd068000059fadcc3059efc679fc824192a4ecb76042', 'clinc150/validation': '6f24511adc7ff42d03bbe0d457a55b3bc334bf14301863c4c5c8510b02aed27d', 'clinc150/test': '05b03fef37ba33303c83f168af1073058a88517fed8fc279e0b2ee61e1e72288', 'clinc150/fewshot': '0243fd72cbd964af306babd636854ace88db0b7aef35beaec46b079fe9ab881d'}
protocol_manifests = {}
audit_rows = []
for dataset, result in [("banking77", banking_protocol), ("clinc150", clinc_protocol)]:
    for split in ["pool", "validation", "test", "fewshot"]:
        frame = result[split][["source_id", "label", "text_sha256"]].copy()
        key = f"{dataset}/{split}"
        payload = frame.to_json(orient="records", force_ascii=False)
        fingerprint = hashlib.sha256(payload.encode("utf-8")).hexdigest()
        assert fingerprint == EXPECTED_MANIFEST_SHA256[key], f"Frozen protocol mismatch: {key}, seed={SEED}, K={K}"
        protocol_manifests[key] = frame
        audit_rows.append({"dataset": dataset, "split": split, "rows": len(frame), "sha256": fingerprint})
display(pd.DataFrame(audit_rows))
for key, frame in protocol_manifests.items():
    print(key, "preview (full table retained in protocol_manifests)")
    display(frame.head(5))
clinc_label_mapping = pd.DataFrame(sorted(clinc_protocol["protocol_meta"]["id_to_name"].items()), columns=["original_id", "intent"])
print("CLINC150 verified mapping:", len(clinc_label_mapping), "unique IDs and intent names")
display(clinc_label_mapping.head(10))
print("All fixed protocol fingerprints verified; no CSV/JSON files exported.")
